Configurando o Ambiente

In [5]:
%idle_timeout 60
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 60 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5


Importações de bibliotecas e iniciando o spark

In [1]:
import sys
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql.functions import col, lit, trim, lower, when, coalesce, sum as spark_sum

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print("Sessao do Glue Notebook pronta")

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 60
Session ID: 7ef5ae1a-cdfb-4ec3-9437-77f2daec2257
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 7ef5ae1a-cdfb-4ec3-9437-77f2daec2257 to get into ready status...
Session 7ef5ae1a-cdfb-4ec3-9437-77f2daec2257 has been created.
Sessao do Glue Notebook pronta


Definindo o caminho para o S3

In [2]:
NOME_BUCKET = "tech-challenge-fase-3-grupo-94"

CAMINHO_SILVER_UNIFICADO = f"s3://{NOME_BUCKET}/silver/state_of_data_unificado/"
CAMINHO_GOLD_LIMPO = f"s3://{NOME_BUCKET}/gold/state_of_data_limpo/"

Leitura da camada Silver

In [3]:
df_consolidado = spark.read.parquet(CAMINHO_SILVER_UNIFICADO)

print(f"Linhas na Silver: {df_consolidado.count()} | Colunas: {len(df_consolidado.columns)}")

Linhas na Silver: 14005 | Colunas: 501


Limpeza, tratamento dos booleanos e seleção de colunas

In [4]:
# 1. Indica as colunas gerais que devem ser mantidas no DataFrame final
colunas_gerais = [
    "id",
    "Idade",
    "Faixa_idade",
    "Genero",
    "Cor_raca_etnia",
    "PCD",
    "experiencia_profissional_prejudicada",
    "aspectos_prejudicados",
    "vive_no_brasil",
    "Estado_onde_mora",
    "uf_onde_mora",
    "Regiao_onde_mora",
    "Mudou_de_Estado",
    "Regiao_de_origem",
    "Nivel_de_Ensino",
    "Area_de_Formacao",
    "Qual_sua_situacao_atual_de_trabalho",
    "Setor",
    "Numero_de_Funcionarios",
    "Gestor",
    "Cargo_como_Gestor",
    "Cargo_Atual",
    "Nivel",
    "Faixa_salarial",
    "Quanto_tempo_de_experiencia_na_area_de_dados_voce_tem",
    "Voce_esta_satisfeito_na_sua_empresa_atual",
    "Qual_o_principal_motivo_da_sua_insatisfacao_com_a_empresa_atual",
    "Atualmente_qual_a_sua_forma_de_trabalho",
    "Qual_a_forma_de_trabalho_ideal_para_voce",
    "Caso_sua_empresa_decida_pelo_modelo_100_presencial_qual_sera_sua_atitude",
    "Qual_o_numero_aproximado_de_pessoas_que_atuam_com_dados_na_sua_empresa_hoje",
    "Quais_desses_papeis_cargos_fazem_parte_do_time_ou_chapter_de_dados_da_sua_empresa",
    "AI_Generativa_e_uma_prioridade_em_sua_empresa",
    "Tipos_de_uso_de_AI_Generativa_e_LLMs_na_empresa",
    "Motivos_que_levam_a_empresa_a_nao_usar_AI_Genrativa_e_LLMs",
    "Atuacao",
    "Quais_das_linguagens_listadas_abaixo_voce_utiliza_no_trabalho_Linguagem",
    "SQL_Linguagem",
    "R_Linguagem",
    "Python_Linguagem",
    "C_C_C_Linguagem",
    "NET_Linguagem",
    "Java_Linguagem",
    "Julia_Linguagem",
    "SAS_Stata_Linguagem",
    "Visual_Basic_VBA_Linguagem",
    "Scala_Linguagem",
    "Matlab_Linguagem",
    "Rust_Linguagem",
    "PHP_Linguagem",
    "JavaScript_Linguagem",
    "DAX_Linguagem",
    "Nao_utilizo_nenhuma_linguagem_Linguagem",
    "Quais_dos_bancos_de_dados_fontes_de_dados_listados_abaixo_voce_utiliza_no_trabalho_BD",
    "MySQL_BD",
    "Oracle_BD",
    "SQL_SERVER_BD",
    "Amazon_Aurora_ou_RDS_BD",
    "DynamoDB_BD",
    "CoachDB_BD",
    "Cassandra_BD",
    "MongoDB_BD",
    "MariaDB_BD",
    "Datomic_BD",
    "S3_BD",
    "PostgreSQL_BD",
    "ElasticSearch_BD",
    "DB2_BD",
    "Microsoft_Access_BD",
    "SQLite_BD",
    "Sybase_BD",
    "Firebase_BD",
    "Vertica_BD",
    "Redis_BD",
    "Neo4J_BD",
    "Google_BigQuery_BD",
    "Google_Firestore_BD",
    "Amazon_Redshift_BD",
    "Amazon_Athena_BD",
    "Snowflake_BD",
    "Databricks_BD",
    "HBase_BD",
    "Presto_BD",
    "Splunk_BD",
    "SAP_HANA_BD",
    "Hive_BD",
    "Firebird_BD",
    "Dentre_as_opcoes_listadas_qual_sua_Cloud_preferida_cloud",
    "Azure_Microsoft_cloud",
    "Amazon_Web_Services_AWS_cloud",
    "Google_Cloud_GCP_cloud",
    "Oracle_Cloud",
    "IBM_cloud",
    "Servidores_On_Premise_Nao_utilizamos_Cloud_cloud",
    "Cloud_Propria_cloud",
    "Ferramenta_de_BI_utilizada_no_dia_a_dia_BI",
    "Microsoft_PowerBI_BI",
    "Qlik_View_Qlik_Sense_BI",
    "Tableau_BI",
    "Metabase_BI",
    "Superset_BI",
    "Redash_BI",
    "Looker_BI",
    "Looker_Studio_Google_Data_Studio_BI",
    "Amazon_Quicksight_BI",
    "Mode_BI",
    "Alteryx_BI",
    "MicroStrategy_BI",
    "IBM_Analytics_Cognos_BI",
    "SAP_Business_Objects_SAP_Analytics_BI",
    "Oracle_Business_Intelligence_BI",
    "Salesforce_Einstein_Analytics_BI",
    "Birst_BI",
    "SAS_Visual_Analytics_BI",
    "Grafana_BI",
    "TIBCO_Spotfire_BI",
    "Pentaho_BI",
    "Fazemos_todas_as_analises_utilizando_apenas_Excel_ou_planilhas_do_google_BI",
    "Nao_utilizo_nenhuma_ferramenta_de_BI_no_trabalho_BI",
    "Qual_o_tipo_de_uso_de_AI_Generativa_e_LLMs_na_empresa",
    "Qual_seu_objetivo_na_area_de_dados",
    "Qual_oportunidade_voce_esta_buscando",
    "Ha_quanto_tempo_voce_busca_uma_oportunidade_na_area_de_dados",
    "Como_tem_sido_a_busca_por_um_emprego_na_area_de_dados",
    "Sua_organizacao_possui_um_Data_Lake",
    "Qual_tecnologia_utilizada_como_plataforma_do_Data_Lake",
    "Sua_organizacao_possui_um_Data_Warehouse",
    "Qual_tecnologia_utilizada_como_plataforma_do_Data_Warehouse",
    "Quais_as_ferramentas_de_gestao_de_Qualidade_de_dados_Metadados_e_catalogo_de_dados_voce_utiliza_no_trabalho",
    "Quais_as_ferramentas_tecnologias_de_ETL_que_voce_utiliza_no_trabalho_como_Data_Analyst_ETL",
    "Scripts_Python_ETL",
    "SQL_Stored_Procedures_ETL",
    "Apache_Airflow_ETL",
    "Apache_NiFi_ETL",
    "Luigi_ETL",
    "AWS_Glue_ETL",
    "Talend_ETL",
    "Pentaho_ETL",
    "Alteryx_ETL",
    "Stitch_ETL",
    "Fivetran_ETL",
    "Google_Dataflow_ETL",
    "Oracle_Data_Integrator_ETL",
    "IBM_DataStage_ETL",
    "SAP_BW_ETL_ETL",
    "SQL_Server_Integration_Services_SSIS_ETL",
    "SAS_Data_Integration_ETL",
    "Qlik_Sense_ETL",
    "Knime_ETL",
    "Databricks_ETL",
    "Nao_utilizo_ferramentas_de_ETL_ETL",
    "ano_pesquisa",
    "data_hora_envio"
]

In [5]:
# 2. Filtrar o DataFrame usando apenas as colunas gerais

# Verifica quais colunas da lista realmente existem, para evitar erros
colunas_existentes = [coluna for coluna in colunas_gerais if coluna in df_consolidado.columns]

# Cria o df_limpo selecionando apenas essas colunas
df_filtrado = df_consolidado.select(*colunas_existentes)

print(f"Colunas selecionadas: {len(df_filtrado.columns)}")

Colunas selecionadas: 153


In [6]:
# 3. Normalizar colunas booleanas
def normalizar_booleano(nome):
    valor = lower(trim(col(nome).cast("string")))
    return (
        when(valor.isin("true", "1.0", "yes", "y"), lit("1"))
        .when(valor.isin("false", "0.0" , "no"), lit("0"))
        .otherwise(col(nome).cast("string"))
    )

for nome_coluna in colunas_gerais:
    if nome_coluna in df_filtrado.columns:
        df_filtrado = df_filtrado.withColumn(nome_coluna, normalizar_booleano(nome_coluna))

In [7]:
# 4. Remover linhas com valores nulos em colunas essenciais para análise
df_gold = df_filtrado.dropna(subset=["Cargo_Atual", "Nivel", "Faixa_salarial", "Idade", "id", "Genero", "Regiao_onde_mora"])

print(f"Total de linhas prontas para analise: {df_gold.count()}")
print(f"Total de colunas prontas para o analise: {len(df_gold.columns)}")

Total de linhas prontas para analise: 9931
Total de colunas prontas para o analise: 153


Evidências analíticas

In [8]:
print("--- ANÁLISE EXPLORATÓRIA ---")

print("\n1. Evolução de Respondentes por Ano:")
df_gold.groupBy("ano_pesquisa").count().orderBy(col("ano_pesquisa").desc()).show()

print("\n2. Top 10 Cargos na Área de Dados:")
df_gold.groupBy("Cargo_Atual").count().orderBy(col("count").desc()).show(10, truncate=False)

print("\n3. Perfil de Senioridade dos Profissionais:")
df_gold.groupBy("Nivel").count().orderBy(col("count").desc()).show(truncate=False)

print("\n4. Distribuição Salarial do Mercado:")
df_gold.groupBy("Faixa_salarial").count().orderBy(col("count").desc()).show(truncate=False)

print("\n5. Modelos de Trabalho Atuais:")
df_gold.groupBy("Atualmente_qual_a_sua_forma_de_trabalho").count().orderBy(col("count").desc()).show(truncate=False)

print("\n6. Satisfação no Trabalho vs. Modelo de Trabalho Atual:")
df_gold.groupBy("Atualmente_qual_a_sua_forma_de_trabalho", "Voce_esta_satisfeito_na_sua_empresa_atual") \
       .count() \
       .orderBy("Atualmente_qual_a_sua_forma_de_trabalho", col("count").desc()) \
       .show(truncate=False)

print("\n7. Inteligência Artificial: Nível de Priorização nas Empresas:")
df_gold.groupBy("AI_Generativa_e_uma_prioridade_em_sua_empresa").count().orderBy(col("count").desc()).show(truncate=False)

--- ANÁLISE EXPLORATÓRIA ---

1. Evolução de Respondentes por Ano:
+------------+-----+
|ano_pesquisa|count|
+------------+-----+
|        2025| 2428|
|        2024| 3723|
|        2023| 3780|
+------------+-----+


2. Top 10 Cargos na Área de Dados:
+-------------------------------------------------------------------+-----+
|Cargo_Atual                                                        |count|
+-------------------------------------------------------------------+-----+
|Analista de Dados/Data Analyst                                     |2404 |
|Cientista de Dados/Data Scientist                                  |1765 |
|Analista de BI/BI Analyst                                          |1104 |
|Engenheiro de Dados/Data Engineer/Data Architect                   |972  |
|Outra Opção                                                        |704  |
|Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect|666  |
|Analista de Negócios/Business Analyst                           

Grava a base limpa em Parquet

In [9]:
df_gold.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(CAMINHO_GOLD_LIMPO)

print(f"Base unificada gravada em {CAMINHO_GOLD_LIMPO}")

Base unificada gravada em s3://tech-challenge-fase-3-grupo-94/gold/state_of_data_limpo/
